# Parkinson image experiment (research only)

This repository contains utilities for an exploratory binary image-classification experiment. **This is not a clinical or diagnostic tool and must not be used for patient care or medical decisions.** No dataset, trained weights, performance result, or external validation is included.

Use an independently sourced dataset only when you have permission and understand its labels. If multiple images can come from one person, provide an anonymized stable group ID and keep every image from that person within a single split. See `README.md` for the full data contract and limitations.

## Validate the data manifest

Install the project first with `pip install -e '.[train]'`. Set `PARKINSON_MANIFEST` to a CSV with exactly `path,label,group_id,split` columns and `PARKINSON_DATA_DIR` to its image root. The allowed labels are `control` and `pd`; the allowed splits are `train`, `validation`, and `test`. Paths must be relative to the image root. Both labels must be present in every split, and group IDs must not cross split boundaries.

Group IDs may be sensitive: keep them local and anonymized. Validation prints aggregate counts only.

In [ ]:
import os
import subprocess
import sys

manifest = os.environ.get('PARKINSON_MANIFEST')
data_root = os.environ.get('PARKINSON_DATA_DIR')
if not manifest or not data_root:
    raise RuntimeError('Set PARKINSON_MANIFEST and PARKINSON_DATA_DIR before running this cell.')
subprocess.run(
    [sys.executable, '-m', 'parkinson.cli', 'validate', '--manifest', manifest, '--data-root', data_root],
    check=True,
)

## Train and evaluate separately

Run the `train` CLI only after reviewing the group-disjoint manifest. It reads `train` and `validation` images for fitting and early stopping; the separate `test` split is never used for model selection. For example (from a terminal):

```bash
python -m parkinson.cli train --manifest "$PARKINSON_MANIFEST" --data-root "$PARKINSON_DATA_DIR" --output-dir ./artifacts/run-1
```

Only after the model and analysis plan are frozen, run `evaluate` once for a descriptive aggregate holdout result:

```bash
python -m parkinson.cli evaluate --manifest "$PARKINSON_MANIFEST" --data-root "$PARKINSON_DATA_DIR" --model-dir ./artifacts/run-1
```

Do not use test results for model selection. Repeated experimentation invalidates this holdout evaluation. These exploratory results cannot establish clinical safety, diagnostic performance, or generalizability.